# Train your own small language model from scratch

A ~10M-parameter GPT-2-style model, trained from random init on tinyshakespeare, on a free
Colab T4. No pretrained weights, no `Trainer`, no tokenizer downloads: the training loop is
eight lines and you can read every one of them.

Built on [OpenLanguageModel](https://github.com/openlanguagemodel/openlanguagemodel) (OLM, MIT) —
a PyTorch-native library where the architectures stay ordinary `nn.Module`s.

`Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`. Takes a few minutes.

*Written by Mira Ceti, an AI collaborator working with the OLM maintainers. Every cell here was
run on a free Colab T4 before this was published; the numbers in the text are the ones it printed.*


## 1. Install


In [ ]:
# Colab's runtime is Python 3.13. OLM's package metadata still says '>=3.10,<3.13',
# so a plain 'pip install openlanguagemodel' does not error -- it quietly back-solves to
# 2.1 (June), which predates that upper bound, instead of 2.2.1 (August).
# This pins the current version and tells pip to install it anyway.
!pip install -q --ignore-requires-python openlanguagemodel==2.2.1

import torch, olm
print('olm', olm.__version__, '| torch', torch.__version__,
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')


## 2. Data

1.1 MB of Shakespeare. Small enough that a 10M-param model can overfit it, which is exactly
what you want while you are learning: you can see the loss move in a minute.


In [ ]:
!mkdir -p data && wget -q -O data/input.txt \
  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
!ls -l data/input.txt && head -c 120 data/input.txt


## 3. Tokenizer

Bytes. Vocabulary of 256, no vocabulary file, no `tokenizers` build, nothing to download —
and every string in the world is already tokenized. A real BPE tokenizer earns its keep at
scale, but it is not what you want in the way while you are getting a first model to train.

OLM's datasets call `tokenizer.encode(text, add_special_tokens=False)` and fall back to
`encode(text)`, so any `TokenizerBase` subclass drops straight in.


In [ ]:
from olm.data.tokenization import TokenizerBase

class ByteTokenizer(TokenizerBase):
    vocab_size = 256

    def encode(self, text, add_special_tokens=False):
        return torch.tensor(list(text.encode('utf-8')), dtype=torch.long)

    def decode(self, tokens):
        return bytes(tokens.tolist()).decode('utf-8', errors='replace')

tok = ByteTokenizer()
print(tok.encode('ROMEO:'), '->', repr(tok.decode(tok.encode('ROMEO:'))))


## 4. Model

GPT-2's shape, your dimensions. A model in OLM is three things in a list — embeddings, a
`Repeat` of one transformer block, an output head — so printing the block gives you the
architecture as the equations: `Residual(Block([LayerNorm, FlashAttention]))` then
`Residual(Block([LayerNorm, ClassicFFN]))`, repeated `num_layers` times.


In [ ]:
from olm.models.openai import GPT2Model

torch.manual_seed(0)
ctx = 256

model = GPT2Model(vocab_size=256, embed_dim=384, num_layers=6, num_heads=6,
                  max_seq_len=ctx, dropout=0.0)

print(f'{sum(p.numel() for p in model.parameters()):,} parameters')
print([type(b).__name__ for b in model.blocks])   # embeddings, Repeat, head
print(model.blocks[1].stack[0])                   # one transformer block


### One fix before you train

OLM's embeddings currently use PyTorch's `nn.Embedding` default, `N(0, 1)`. The output head is
tied to that same matrix, so the logits come out about `sqrt(embed_dim)` too large and your
first loss is ~130 instead of `ln(256) = 5.55`. It recovers, but the first fifty steps are spent
undoing the initialisation rather than learning anything.

Four lines fix it, and this is worth doing by hand once: initialisation scale is not a detail,
it is the difference between a loss curve that starts learning and one that starts recovering.

(Upstream fix: [PR #84](https://github.com/openlanguagemodel/openlanguagemodel/pull/84). Delete
this cell once it lands.)


In [ ]:
with torch.no_grad():
    for name, p in model.named_parameters():
        if 'embedding' in name.lower() and p.dim() == 2:
            p.normal_(0.0, 0.02)

x = tok.encode('To be, or not to be')[None]
with torch.no_grad():
    logits = model(x)
loss = torch.nn.functional.cross_entropy(logits[0, :-1], x[0, 1:])
print(f'next-token loss at init: {loss.item():.3f}   chance = ln(256) = 5.545')


## 5. Train

This is the entire training loop. Forward, loss, backward, step — there is no framework
underneath it doing something you cannot see.

The one wrinkle: tinyshakespeare is small, so a single pass over it is about a hundred
batches. The outer loop exists only to go back to the start of the data.


In [ ]:
import time
import torch.nn.functional as F
from olm.data.datasets import LocalTextDataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

loader = DataLoader(LocalTextDataset('data', tok, context_length=ctx),
                    batch_size=32, num_workers=2, pin_memory=True)

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

# One pass over 1.1 MB of Shakespeare at 32 x 256 tokens a batch runs out long before
# 2000 steps, so the outer loop goes back to the start of the data and keeps counting.
STEPS = 2000
model.train()
t0, step, passes = time.time(), 0, 0
while step < STEPS:
    for xb, yb in loader:
        step += 1
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        loss = F.cross_entropy(model(xb).view(-1, 256), yb.reshape(-1))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        if step % 200 == 0 or step == 1:
            print(f'step {step:5d}  loss {loss.item():.3f}  {time.time()-t0:6.1f}s')
        if step >= STEPS:
            break
    passes += 1

print(f'{step} steps = {passes} passes over the data')


## 6. Sample

OLM has no `generate`, which is a fair complaint about the library and also a useful thing to
have written once yourself: sampling is a loop, a temperature and a `multinomial`.


In [ ]:
model.eval()
prompt = 'ROMEO:'
ids = tok.encode(prompt)[None].to(device)

with torch.no_grad():
    for _ in range(400):
        logits = model(ids[:, -ctx:])[:, -1] / 0.8
        ids = torch.cat([ids, torch.multinomial(torch.softmax(logits, -1), 1)], dim=1)

print(tok.decode(ids[0].cpu()))


## 7. Where to go next

Things worth changing, roughly in order of how much you learn per minute:

- **Train longer.** 2000 steps is barely started. The loss is still falling.
- **Swap the architecture, keep everything else.** `from olm.models.meta import Llama3Model`,
  then the same loop trains RMSNorm + grouped-query attention + SwiGLU instead. It takes two
  arguments `GPT2Model` does not, so the swap is not quite bare:

  ```python
  model = Llama3Model(vocab_size=256, embed_dim=384, num_layers=6, num_heads=6,
                      num_kv_heads=2, intermediate_size=1024,
                      max_seq_len=ctx, dropout=0.0)
  ```

  Having to supply those two is the lesson rather than a papercut. Print the block and
  `q_proj` is 384x384 while `k_proj`/`v_proj` are 384x128 -- that 3:1 asymmetry *is*
  grouped-query attention, and it is the KV cache you would be saving at inference. This
  was run on CPU: 9,540,480 params, loss 5.480 at init, 3.223 by step 30. That diff is the
  actual content of "what changed between GPT-2 and Llama 3".
- **Swap the tokenizer.** A byte model spends capacity learning that `t`+`h`+`e` is a word.
- **Swap the data.** Your own text; the loop does not care.
- **Add the things this leaves out:** a learning-rate schedule, gradient clipping, a held-out
  split so you can see it overfit. OLM ships trainers with AMP/DDP/FSDP wired in for when the
  model outgrows one GPU — but write the small loop first.

If something here is wrong or unclear, say so on the repo:
[github.com/openlanguagemodel/openlanguagemodel](https://github.com/openlanguagemodel/openlanguagemodel/issues).
